# M2: BiLSTM with attention on Kaggle (multi-task, tone-only, type-only)

This notebook runs the three BiLSTM runs from Phase 2 on a Kaggle GPU. All the
training code lives in `scripts/train_char.py`; the notebook only calls it with
Kaggle's file paths. That way the Mac results and the Kaggle results come from
exactly the same code.

**Checkpoints.** After each seed finishes, its best model is saved as a `.pt` file.
If the session stops halfway, run all cells again: seeds that already have a
checkpoint are skipped, and training continues with the next one.

## One-time setup on Kaggle
1. **Upload the data as a PRIVATE dataset.** Kaggle → *Datasets* → *New Dataset*.
   Upload these three files from the project folder:
   - `data/train_clean.csv`
   - `data/dev_multitask.csv`
   - `scripts/train_char.py`

   Set visibility to **Private**. Do **not** upload any test file: the test set is
   scored only once, at the very end, on the Mac with `scripts/score_test.py`.
2. **Import this notebook.** Kaggle → *Create* → *New Notebook* → *File* → *Import Notebook*
   → choose `notebooks/m2_bilstm_kaggle.ipynb`.
3. **Attach the dataset.** Right-hand panel → *Add Input* → pick your dataset.
4. **Turn on the GPU.** *Settings* → *Accelerator* → *GPU T4 x2* (or *GPU P100*).
5. *(Optional)* *Settings* → *Persistence* → *Files only*, so the checkpoints in
   `/kaggle/working` survive a restart of an interactive session.
6. **Run.** First run with `QUICK_TEST = True` (about one minute) to check that everything
   works. Then set it to `False` and click *Save Version* → *Save & Run All (Commit)*.
   That runs in the background, even if you close the browser.

## Bringing the results back to the Mac
When the version has finished: open it → *Output* tab → download `m2_bilstm_outputs.zip`.
Unzip it **inside the project folder** (`tulu-hope-speech/`). It contains:
- `results/m2_bilstm_multi.json`, `..._coarse.json`, `..._fine.json`: the scores
- `results/logs/`: the full training printouts
- `checkpoints/`: the saved models (git ignores these, because they are large)

In [ ]:
# Step 1: find the input files and choose where outputs go.
from pathlib import Path   # Path handles file paths, e.g. Path("a") / "b" means "a/b"
import torch

ON_KAGGLE = Path("/kaggle/input").exists()

# Set to True for a 1-minute check (1 seed, 2 epochs). Set to False for the real run.
QUICK_TEST = False

if ON_KAGGLE:
    def find(name):
        # rglob searches every sub-folder of /kaggle/input for a file with this name,
        # so it doesn't matter what you named your dataset.
        matches = sorted(Path("/kaggle/input").rglob(name))
        if not matches:
            raise FileNotFoundError(f"{name} not found. Did you attach the dataset (Add Input)?")
        return matches[0]

    SCRIPT = find("train_char.py")
    TRAIN = find("train_clean.csv")
    DEV = find("dev_multitask.csv")
    OUT = Path("/kaggle/working/m2_outputs")   # /kaggle/working is the folder Kaggle keeps
else:
    # Running on our own machine, from the notebooks/ folder: use the normal project paths.
    SCRIPT = Path("../scripts/train_char.py")
    TRAIN = Path("../data/train_clean.csv")
    DEV = Path("../data/dev_multitask.csv")
    OUT = Path("..")

RESULTS = OUT / "results"
CKPT = OUT / "checkpoints"
LOGS = RESULTS / "logs"
LOGS.mkdir(parents=True, exist_ok=True)   # parents=True also creates the folders above it

MODEL = "bilstm"
EXTRA = "--seeds 1 --epochs 2" if QUICK_TEST else ""   # extra options for a quick test

print("script :", SCRIPT)
print("train  :", TRAIN)
print("dev    :", DEV)
print("outputs:", OUT)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")

## Step 2: the three runs
Each cell runs `train_char.py` once (5 seeds). A line starting with `!` is a
shell command, not Python; `{SCRIPT}` etc. insert the Python variables from
Step 1. `python -u` prints each line straight away, and `| tee file` shows the
output here and also saves it to a log file.

In [ ]:
# multi-task: shared encoder, both heads trained
!python -u {SCRIPT} --model {MODEL} --task multi --train {TRAIN} --dev {DEV} --out_dir {RESULTS} --ckpt_dir {CKPT} {EXTRA} 2>&1 | tee {LOGS}/m2_{MODEL}_multi.log

In [ ]:
# single-task control: only the tone head is trained
!python -u {SCRIPT} --model {MODEL} --task coarse --train {TRAIN} --dev {DEV} --out_dir {RESULTS} --ckpt_dir {CKPT} {EXTRA} 2>&1 | tee {LOGS}/m2_{MODEL}_coarse.log

In [ ]:
# single-task control: only the hope-type head is trained
!python -u {SCRIPT} --model {MODEL} --task fine --train {TRAIN} --dev {DEV} --out_dir {RESULTS} --ckpt_dir {CKPT} {EXTRA} 2>&1 | tee {LOGS}/m2_{MODEL}_fine.log

In [ ]:
# Step 3: summary of the three runs, and a check for classes stuck at 0.000 F1.
import json
import pandas as pd

rows = []
for task in ["multi", "coarse", "fine"]:
    path = RESULTS / f"m2_{MODEL}_{task}.json"
    if not path.exists():
        print("missing:", path)
        continue
    r = json.loads(path.read_text())
    rows.append({"task": task,
                 "tone F1": r["tone_mean"], "tone std": r["tone_std"],
                 "type F1": r["type_mean"], "type std": r["type_std"],
                 "best epochs": r["best_epochs"]})

    # Look at every seed, not only the best one.
    for i, seed_scores in enumerate(r["per_class_f1_by_seed"]):
        for head, classes in seed_scores.items():
            for name, f1 in classes.items():
                if f1 == 0.0:
                    print(f"WARNING {task}: seed {r['seeds'][i]} has 0.000 F1 for {head} class '{name}'")

print("M1 baseline to beat: tone 0.579, type 0.400")
pd.DataFrame(rows).round(3)

In [ ]:
# Step 4 (Kaggle only): pack results + checkpoints into one zip to download from the Output tab.
import shutil

if ON_KAGGLE:
    zip_path = shutil.make_archive("/kaggle/working/m2_bilstm_outputs", "zip", root_dir=OUT)
    print("created", zip_path, "- download it from the Output tab and unzip inside tulu-hope-speech/")
else:
    print("Running locally: results are already in the project's results/ folder.")